# Marketing production - Autocall France Juillet 2018

Ce notebook lance la generation complete de la presentation HTML marketing pour l'Autocall France Juillet 2018.

### Import librairies

In [1]:
from pathlib import Path
import os
import sys
import json
import warnings
import matplotlib

### Gestion des chemins

In [2]:
def find_project_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / "src" / "marketing_production").exists():
            return path
    raise FileNotFoundError("Impossible de trouver la racine du projet structured_products")


PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"
ASSET_DIR = SRC_DIR / "marketing_production" / "assets"
INSTRUMENT_NOTEBOOK = SRC_DIR / "marketing_production" / "0_product_terms_instrument.ipynb"
OUTPUT_HTML = PROJECT_ROOT / "HTML" / "Autocall_France_Juillet_2018.html"

ASSET_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / ".matplotlib-cache"))

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

PROJECT_ROOT

PosixPath('/Users/arthurlenet/Desktop/structured_products')

### 1. Rafraichissement des visuels instrument

Cette etape relance les cellules du notebook instrument qui exportent les images PNG necessaires a la presentation.

In [3]:
def refresh_instrument_assets(notebook_path=INSTRUMENT_NOTEBOOK):
    """Execute setup cells, then the instrument cells that export PNG assets used by the deck."""
    matplotlib.use("Agg", force=True)

    notebook = json.loads(notebook_path.read_text(encoding="utf-8"))
    namespace = {"__name__": "__main__"}
    executed_cells = []
    found_export_cell = False

    for cell_index, cell in enumerate(notebook["cells"]):
        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))
        exports_slide_asset = "fig.savefig(" in source and (
            "PRODUCT_TERMS_IMAGE_PATH" in source
            or "PRODUCT_DATES_IMAGE_PATH" in source
            or "PRODUCT_REFERENCE_IMAGE_PATH" in source
        )
        if found_export_cell and not exports_slide_asset:
            continue

        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", message="FigureCanvasAgg is non-interactive.*")
            exec(compile(source, f"{notebook_path}#cell-{cell_index}", "exec"), namespace)

        import matplotlib.pyplot as plt

        plt.close("all")
        if exports_slide_asset:
            found_export_cell = True
            executed_cells.append(cell_index)

    if not executed_cells:
        raise RuntimeError(f"Aucune cellule d'export PNG trouvee dans {notebook_path}")

    return executed_cells


executed_cells = refresh_instrument_assets()
print(f"Assets instrument rafraichis depuis les cellules: {executed_cells}")

Assets instrument rafraichis depuis les cellules: [4, 5, 6]


### 2. Verification des assets attendus

Cette etape controle que toutes les images necessaires a la presentation existent avant de lancer la generation finale.

In [4]:
EXPECTED_ASSETS = [
    ASSET_DIR / "product_terms_snapshot.png",
    ASSET_DIR / "product_dates_snapshot.png",
    ASSET_DIR / "product_reference_snapshot.png",
    ASSET_DIR / "underlying_performance_snapshot.png",
    ASSET_DIR / "drawdown_snapshot.png",
    ASSET_DIR / "underlying_snapshot.png",
    ASSET_DIR / "autocall_snapshot.png",
    ASSET_DIR / "capital_barrier_snapshot.png",
    ASSET_DIR / "autocall_monitoring_snapshot.png",
]

missing_assets = [path for path in EXPECTED_ASSETS if not path.exists()]
if missing_assets:
    formatted = "\n".join(f"- {path}" for path in missing_assets)
    raise FileNotFoundError(f"Assets manquants pour la presentation:\n{formatted}")

for asset_path in EXPECTED_ASSETS:
    print(f"OK - {asset_path.relative_to(PROJECT_ROOT)} ({asset_path.stat().st_size / 1024:.0f} KB)")


OK - src/marketing_production/assets/product_terms_snapshot.png (172 KB)
OK - src/marketing_production/assets/product_dates_snapshot.png (186 KB)
OK - src/marketing_production/assets/product_reference_snapshot.png (145 KB)
OK - src/marketing_production/assets/underlying_performance_snapshot.png (310 KB)
OK - src/marketing_production/assets/drawdown_snapshot.png (202 KB)
OK - src/marketing_production/assets/underlying_snapshot.png (133 KB)
OK - src/marketing_production/assets/autocall_snapshot.png (136 KB)
OK - src/marketing_production/assets/capital_barrier_snapshot.png (102 KB)
OK - src/marketing_production/assets/autocall_monitoring_snapshot.png (295 KB)


### 3. Generation de la presentation HTML

Cette etape recharge le module de production marketing et genere le fichier HTML final a partir des assets verifies.

In [5]:
for module_name in list(sys.modules):
    if module_name == "marketing_production" or module_name.startswith("marketing_production."):
        del sys.modules[module_name]

from marketing_production import build_presentation

output_path = build_presentation.generate(output_path=OUTPUT_HTML)
print(f"Presentation HTML generee: {output_path}")
output_path

Presentation HTML generee: /Users/arthurlenet/Desktop/structured_products/HTML/Autocall_France_Juillet_2018.html


PosixPath('/Users/arthurlenet/Desktop/structured_products/HTML/Autocall_France_Juillet_2018.html')